In [ ]:
## Load necessary modules
# %load_ext watermark

import asf_search as asf
import geopandas as gpd
import pandas as pd

import numpy as np
from netrc import netrc
from subprocess import Popen
from platform import system
from getpass import getpass
import folium
import datetime as dt
from shapely.geometry import box
from shapely.geometry import Point
import shapely.wkt as wkt
import rioxarray
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import cartopy.crs as ccrs
import xarray as xr
from rasterio.plot import show


# %watermark --iversions

import os

In [ ]:
## Avoid lots of these warnings printing to notebook from asf_search
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Generates authentication token
# Asks for your Earthdata username and password for first time, if netrc does not exists.

urs = 'urs.earthdata.nasa.gov'    # Earthdata URL endpoint for authentication
prompts = ['Enter NASA Earthdata Login Username: ',
           'Enter NASA Earthdata Login Password: ']

# Determine the OS (Windows machines usually use an '_netrc' file)
netrc_name = "_netrc" if system()=="Windows" else ".netrc"

# Determine if netrc file exists, and if so, if it includes NASA Earthdata Login Credentials
try:
    netrcDir = os.path.expanduser(f"~/{netrc_name}")
    netrc(netrcDir).authenticators(urs)[0]

# Below, create a netrc file and prompt user for NASA Earthdata Login Username and Password
except FileNotFoundError:
    homeDir = os.path.expanduser("~")
    Popen('touch {0}{2} | echo machine {1} >> {0}{2}'.format(homeDir + os.sep, urs, netrc_name), shell=True)
    Popen('echo login {} >> {}{}'.format(getpass(prompt=prompts[0]), homeDir + os.sep, netrc_name), shell=True)
    Popen('echo \'password {} \'>> {}{}'.format(getpass(prompt=prompts[1]), homeDir + os.sep, netrc_name), shell=True)
    # Set restrictive permissions
    Popen('chmod 0600 {0}{1}'.format(homeDir + os.sep, netrc_name), shell=True)

    # Determine OS and edit netrc file if it exists but is not set up for NASA Earthdata Login
except TypeError:
    homeDir = os.path.expanduser("~")
    Popen('echo machine {1} >> {0}{2}'.format(homeDir + os.sep, urs, netrc_name), shell=True)
    Popen('echo login {} >> {}{}'.format(getpass(prompt=prompts[0]), homeDir + os.sep, netrc_name), shell=True)
    Popen('echo \'password {} \'>> {}{}'.format(getpass(prompt=prompts[1]), homeDir + os.sep, netrc_name), shell=True)


In [ ]:
## Enter user-defined parameters
aoi = wkt.loads("POLYGON((-114.929 42.651,-111.9262 42.651,-111.9262 43.6462,-114.929 43.6462,-114.929 42.651))")
wavelength = 0.056
orbitPass = "DESCENDING"
start_date = dt.datetime.fromisoformat('2025-08-08 00:00:00')         #'YYYY-MM-DD HH:MM:SS'
end_date = dt.datetime.fromisoformat('2025-08-09 00:00:00')         #'YYYY-MM-DD HH:MM:SS'

savedir = './snakeriver'

In [ ]:
## Search for OPERA CSLC data in ASF DAAC
search_params = dict(
    intersectsWith= aoi.wkt,
    dataset='OPERA-S1',
    processingLevel='RTC',
    flightDirection = orbitPass,
    start=start_date,
    end=end_date
)

## Return results
results = asf.search(**search_params)
len(results)

In [ ]:
## Display the queried data
gf = gpd.GeoDataFrame.from_features(results.geojson(), crs='EPSG:4326')
gf.head()

In [ ]:
gf.keys()

In [ ]:
## Filter data based on specified track number
# gf = gf[gf.pathNumber==pathNumber]
rtc_df = gf[['operaBurstID', 'fileID', 'polarization', 'startTime', 'stopTime', 'url', 'additionalUrls', 'geometry']]
rtc_df['startTime'] = pd.to_datetime(rtc_df.startTime).dt.date
rtc_df['stopTime'] = pd.to_datetime(rtc_df.stopTime).dt.date
rtc_df = rtc_df.drop_duplicates(subset=['operaBurstID', 'startTime'], ignore_index=True)
rtc_df

In [ ]:
rtc_df = rtc_df.sort_values(by=["startTime"], ignore_index=True)
rtc_df

In [ ]:
## Basemap function for Folium
def getbasemaps():
    # Add custom base maps to folium
    basemaps = {
        'Google Satellite Hybrid': folium.TileLayer(
            tiles = 'https://mt1.google.com/vt/lyrs=y&x={x}&y={y}&z={z}',
            attr = 'Google',
            name = 'Google Satellite',
            overlay = True,
            control = True,
            # opacity = 0.8,
            show = False
        ),
        'Esri Satellite': folium.TileLayer(
            tiles = 'https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}',
            attr = 'Esri',
            name = 'Esri Satellite',
            overlay = True,
            control = True,
            #opacity = 0.8,
            show = False
        )
    }
    return basemaps

In [ ]:
# Interactive map to visualize the boundaries of the selected OPERA CSLCs
m = folium.Map(location=[aoi.centroid.y, aoi.centroid.x], zoom_start=8, tiles="CartoDB positron")

# Add custom basemaps
basemaps = getbasemaps()
for basemap in basemaps:
    basemaps[basemap].add_to(m)

# layer Control
m.add_child(folium.LayerControl())

## RLE sites
for _, r in rtc_df.iterrows():
    sim_geo = gpd.GeoSeries(r["geometry"]).simplify(tolerance=0.001)
    geo_j = sim_geo.to_json()
    geo_j = folium.GeoJson(data=geo_j, style_function=lambda x: {"color": "red"}, control=False)
    folium.Popup(r["operaBurstID"]).add_to(geo_j)
    geo_j.add_to(m)

m

In [ ]:
## Download the relevant OPERA RTCs to a temporary "rtc" folder 
os.makedirs(f'{savedir}/rtc', exist_ok='True')
asf.download_urls(urls=rtc_df.url, path=f'{savedir}/rtc', processes=10)
asf.download_urls(urls=rtc_df.additionalUrls.explode(ignore_index=True), path=f'{savedir}/rtc', processes=10)

In [ ]:
def read_rtc(tiffile):
    rtc = xr.open_dataset(tiffile)
    epsg = rtc.rio.crs.to_proj4()

    return rtc, epsg

In [ ]:
bbox = [aoi.bounds[0], aoi.bounds[2], aoi.bounds[1], aoi.bounds[3]]

# Merge the RTC bursts

In [ ]:
import rasterio
from rasterio.merge import merge
import glob
import os

def read_and_merge_rtc(rtc_path, output_dir):
    vv_files = glob.glob(f"{rtc_path}/*_VV.tif")
    vh_files = glob.glob(f"{rtc_path}/*_VH.tif")

    # VV
    src_files_to_mosaic = [rasterio.open(vv_file) for vv_file in vv_files]
    VV_, out_transform = merge(src_files_to_mosaic, nodata=np.nan)
    for src in src_files_to_mosaic:
        src.close()

    # VH
    src_files_to_mosaic = [rasterio.open(vh_file) for vh_file in vh_files]
    VH_, out_transform = merge(src_files_to_mosaic, nodata=np.nan)
    for src in src_files_to_mosaic:
        src.close()

    # Convert to dB
    VV = 10*np.log10(VV_)
    VH = 10*np.log10(VH_)

    # Get ratio
    ratio_VV_VH = VV - VH

    # Save VV, VH, and ratio as a multi-band TIFF
    def save_as_multiband_tiff(vv_data, vh_data, ratio_data, filename):
        """ Helper function to save VV, VH, and ratio as multi-band GeoTIFF """
        with rasterio.open(vv_files[0]) as src:  # Use the first VV file to get metadata
            meta = src.meta.copy()
            meta.update({
                "driver": "GTiff",
                "count": 3,  # Three bands: VV, VH, and ratio
                "dtype": "float32",
                "crs": src.crs,
                "transform": out_transform,
                "height": VV_.shape[1],
                "width": VV_.shape[2],
            })

            output_file = os.path.join(output_dir, filename)
            with rasterio.open(output_file, "w", **meta) as dest:
                dest.write(vv_data, 1)  # Write VV to band 1
                dest.write(vh_data, 2)  # Write VH to band 2
                dest.write(ratio_data, 3)  # Write ratio to band 3            
        return meta

    # Save as multi-band TIFF
    meta = save_as_multiband_tiff(VV.squeeze(), VH.squeeze(), ratio_VV_VH.squeeze(), "merged_VV_VH_ratio.tif")

    return VV, VH, ratio_VV_VH, meta

In [ ]:
# Get the VV, VH, and VV/VH ratio. Merge the bursts.
VV, VH, ratio_VV_VH, meta_rtc = read_and_merge_rtc(f"{savedir}/rtc", f"{savedir}/rtc")

In [ ]:
# Plot them all
fig, ax = plt.subplots(1, 3, figsize=(15, 9))

# Plot VV
cax1 = ax[0].imshow(VV.squeeze(), cmap='hot', interpolation=None, origin='upper', extent=bbox, 
                    vmin=np.nanpercentile(VV, 2), vmax=np.nanpercentile(VV, 98))
cbar1 = fig.colorbar(cax1, ax=ax[0], orientation='vertical', fraction=0.01, pad=0.02)
cbar1.set_label('VV', fontsize=12)
ax[0].set_xlabel('Longitude')
ax[0].set_ylabel('Latitude')
ax[0].set_title(f'VV - {pd.to_datetime(start_date).date()}', fontsize=12)

# Plot VH
cax2 = ax[1].imshow(VH.squeeze(), cmap='hot', interpolation=None, origin='upper', extent=bbox, 
                    vmin=np.nanpercentile(VH, 2), vmax=np.nanpercentile(VH, 98))
cbar2 = fig.colorbar(cax2, ax=ax[1], orientation='vertical', fraction=0.01, pad=0.02)
cbar2.set_label('VH', fontsize=12)
ax[1].set_xlabel('Longitude')
ax[1].set_ylabel('Latitude')
ax[1].set_title(f'VH - {pd.to_datetime(start_date).date()}', fontsize=12)

# Plot VV/VH Ratio
cax3 = ax[2].imshow(ratio_VV_VH.squeeze(), cmap='hot', interpolation=None, origin='upper', extent=bbox, 
                    vmin=np.nanpercentile(ratio_VV_VH, 2), vmax=np.nanpercentile(ratio_VV_VH, 98))
cbar3 = fig.colorbar(cax3, ax=ax[2], orientation='vertical', fraction=0.01, pad=0.02)
cbar3.set_label('VV/VH Ratio', fontsize=12)
ax[2].set_xlabel('Longitude')
ax[2].set_ylabel('Latitude')
ax[2].set_title(f'VV/VH Ratio - {pd.to_datetime(start_date).date()}', fontsize=12)

plt.tight_layout()  # Adjust subplots to avoid overlap

In [ ]:
# Visualize as RGB
def open_geotiff_as_rgb(file_path):
    with rasterio.open(file_path) as src:
        # Read all bands (VV -> Band 1, VH -> Band 2, Ratio -> Band 3)
        vv_data = src.read(1)  # Band 1: VV
        vh_data = src.read(2)  # Band 2: VH
        ratio_data = src.read(3)  # Band 3: VV/VH Ratio

        # Debugging: Print min/max values of the bands to understand their ranges
        print("VV min/max:", np.nanmin(vv_data), np.nanmax(vv_data))
        print("VH min/max:", np.nanmin(vh_data), np.nanmax(vh_data))
        print("Ratio min/max:", np.nanmin(ratio_data), np.nanmax(ratio_data))
        
        # Normalize the bands to 0-255 for RGB visualization using np.nanmin and np.nanmax
        vv_scaled = np.interp(vv_data, (np.nanpercentile(vv_data,2), np.nanpercentile(vv_data, 98)), (0, 255)).astype(np.uint8)
        vh_scaled = np.interp(vh_data, (np.nanpercentile(vh_data,2), np.nanpercentile(vh_data, 98)), (0, 255)).astype(np.uint8)
        ratio_scaled = np.interp(ratio_data, (np.nanpercentile(ratio_data,2), np.nanpercentile(ratio_data, 98)), (0, 255)).astype(np.uint8)

        # Combine the bands into an RGB image (VV -> Red, VH -> Green, Ratio -> Blue)
        rgb_image = np.stack((vv_scaled, vh_scaled, ratio_scaled), axis=-1)

    return rgb_image

In [ ]:
# Sanity Check
file_path = f"{savedir}/rtc/merged_VV_VH_ratio.tif"  # Path to the multi-band GeoTIFF
print(file_path)

# Open the GeoTIFF and get the RGB image
gamma0_rgb = open_geotiff_as_rgb(file_path)

# Plot the RGB image
fig, ax = plt.subplots(figsize=(12, 12))
cax = ax.imshow(gamma0_rgb, cmap='jet', vmin=np.nanpercentile(gamma0_rgb, 2), vmax=np.nanpercentile(gamma0_rgb, 98))
ax.set_title("VV (Red), VH (Green), VV/VH Ratio (Blue)", fontsize=12)
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
ax.axis('off')  # Hide axis labels

# cbar = fig.colorbar(cax, ax=ax, orientation='vertical', fraction=0.03, pad=0.04)  # Adjust the fraction and pad to reduce size
# cbar.set_label('Normalized RGB Value', fontsize=12)
# cbar.set_ticks([0, 255])  # Set the colorbar ticks to min and max values

# Convert RTC Gamma0 (Terrain-LOS and brightness corrected) to Sigma0 (Brightness + Local Terrain corrected)

In [ ]:
## Search for OPERA RTC Static data in ASF DAAC
search_params = dict(
    intersectsWith= aoi.wkt,
    dataset='OPERA-S1',
    processingLevel='RTC-STATIC',
    flightDirection = orbitPass,
    # start=start_date,
)

## Return results
results = asf.search(**search_params)
len(results)

In [ ]:
## Display the queried data
gf = gpd.GeoDataFrame.from_features(results.geojson(), crs='EPSG:4326')
gf

In [ ]:
## Filter data and drop duplicates
rtc_static_df = gf[['operaBurstID', 'fileID', 'polarization', 'startTime', 'stopTime', 'url', 'additionalUrls', 'geometry']]
rtc_static_df['startTime'] = pd.to_datetime(rtc_static_df.startTime).dt.date
rtc_static_df['stopTime'] = pd.to_datetime(rtc_static_df.stopTime).dt.date
rtc_static_df = rtc_static_df.drop_duplicates(subset=['operaBurstID', 'startTime'], ignore_index=True)
rtc_static_df = rtc_static_df.sort_values(by=["startTime"], ignore_index=True)
rtc_static_df

In [ ]:
## Get only static layers that correspond to the actual rtc burst_ids that we stitched together earlier
rtc_static_df_filtered = rtc_static_df[rtc_static_df['operaBurstID'].isin(rtc_df['operaBurstID'])].reset_index(drop=True)
# Print one example
rtc_static_df_filtered['additionalUrls'][0][-1]

In [ ]:
## Take the urls of the all the *gamma0_to_sigma0.tif
gamma0_sigma0_files = rtc_static_df_filtered['additionalUrls'].apply(lambda x: x[-1] if isinstance(x, (list, str)) else None)
gamma0_sigma0_files

In [ ]:
## Download the relevant OPERA RTC Static Layers to a temporary "rtc" folder 
os.makedirs(f'{savedir}/rtc', exist_ok='True')
asf.download_urls(urls=rtc_static_df_filtered.url, path=f'{savedir}/rtc', processes=10)
asf.download_urls(urls=gamma0_sigma0_files, path=f'{savedir}/rtc', processes=10)

In [ ]:
def read_and_merge_rtc_static(rtc_path, output_dir, output_filename="merged_gamma0_to_sigma0.tif"):
    # Get list of all RTC files
    g2s_files = glob.glob(f"{rtc_path}")

    # Open all the files for merging
    for file in g2s_files:
        with rasterio.open(file) as src:
            width = src.width
            height = src.height
            print(f"File: {file}, Shape: ({height}, {width})")  # Print shape as (rows, columns)
 
    src_files_to_mosaic = [rasterio.open(f) for f in g2s_files]
    
    # Merge all files into one
    gamma0_to_sigma, out_transform = merge(src_files_to_mosaic, nodata=np.nan)
    gamma0_to_sigma = np.squeeze(gamma0_to_sigma)
    
    # Close the opened files
    for src in src_files_to_mosaic:
        src.close()
    
    # Get metadata from the first file for proper GeoTIFF saving
    with rasterio.open(g2s_files[0]) as src:
        meta = src.meta.copy()
        meta.update({
            "driver": "GTiff",
            "count": 1,  # We are saving as a single-band image
            "dtype": "float32",  # Set dtype to float32 for continuous data
            "crs": src.crs,  # Coordinate reference system
            "transform": out_transform,
            "height": gamma0_to_sigma.shape[0],
            "width": gamma0_to_sigma.shape[1],
        })

    # Ensure output directory exists
    os.makedirs(output_dir, exist_ok=True)
    
    # Save the result as a GeoTIFF file
    output_path = os.path.join(output_dir, output_filename)
    with rasterio.open(output_path, "w", **meta) as dst:
        dst.write(gamma0_to_sigma, 1)  # Write the data to the first (and only) band

    print(f"GeoTIFF saved to {output_path}")    
    return gamma0_to_sigma, meta

In [ ]:
# Read, merge, plot the gamma0_to_sigma00 static layer.
gamma0_to_sigma0, meta_static = read_and_merge_rtc_static(f"{savedir}/rtc/*_rtc_anf_gamma0_to_sigma0.tif", f"{savedir}/rtc")

# Plot
fig, ax = plt.subplots(figsize=(12, 12))
cax = ax.imshow(gamma0_to_sigma0, cmap='jet', vmin=np.nanpercentile(gamma0_to_sigma0, 2), vmax=np.nanpercentile(gamma0_to_sigma0, 98))
ax.set_title("Gamm0 to Sigma0 Factor", fontsize=12)
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
ax.axis('off')  # Hide axis labels

In [ ]:
# Plot each polarization converted from gamma0 to sigma0
fig, ax = plt.subplots(1, 3, figsize=(15, 9))

# Gamma0 to Sigma0 conversion
VV_sigma0 = VV[0] * gamma0_to_sigma0
VH_sigma0 = VH[0] * gamma0_to_sigma0
ratio_VVVH_sigma0 = VV_sigma0 - VH_sigma0

# Plot VV
cax1 = ax[0].imshow(VV_sigma0, cmap='hot', interpolation=None, origin='upper', extent=bbox, 
                    vmin=np.nanpercentile(VV_sigma0, 2), vmax=np.nanpercentile(VV_sigma0, 98))
cbar1 = fig.colorbar(cax1, ax=ax[0], orientation='vertical', fraction=0.01, pad=0.02)
cbar1.set_label('VV_sigma0', fontsize=12)
ax[0].set_xlabel('Longitude')
ax[0].set_ylabel('Latitude')
ax[0].set_title(f'VV_sigma0 - {pd.to_datetime(start_date).date()}', fontsize=12)

# Plot VH
cax2 = ax[1].imshow(VH_sigma0, cmap='hot', interpolation=None, origin='upper', extent=bbox, 
                    vmin=np.nanpercentile(VH_sigma0, 2), vmax=np.nanpercentile(VH_sigma0, 98))
cbar2 = fig.colorbar(cax2, ax=ax[1], orientation='vertical', fraction=0.01, pad=0.02)
cbar2.set_label('VH_sigma0', fontsize=12)
ax[1].set_xlabel('Longitude')
ax[1].set_ylabel('Latitude')
ax[1].set_title(f'VH_sigma0 - {pd.to_datetime(start_date).date()}', fontsize=12)

# Plot VV/VH Ratio
cax3 = ax[2].imshow(ratio_VVVH_sigma0, cmap='hot', interpolation=None, origin='upper', extent=bbox, 
                    vmin=np.nanpercentile(ratio_VVVH_sigma0, 2), vmax=np.nanpercentile(ratio_VVVH_sigma0, 98))
cbar3 = fig.colorbar(cax3, ax=ax[2], orientation='vertical', fraction=0.01, pad=0.02)
cbar3.set_label('VV/VH Ratio (Sigma0)', fontsize=12)
ax[2].set_xlabel('Longitude')
ax[2].set_ylabel('Latitude')
ax[2].set_title(f'VV/VH Ratio Sigma0 - {pd.to_datetime(start_date).date()}', fontsize=12)

plt.tight_layout()  # Adjust subplots to avoid overlap

In [ ]:
# Save VV, VH, and ratio as a multi-band TIFF
def save_as_multiband_tiff(vv_data, vh_data, ratio_data, output_dir, filename, meta):
        output_file = os.path.join(output_dir, filename)
        with rasterio.open(output_file, "w", **meta) as dest:
            dest.write(vv_data, 1)  # Write VV to band 1
            dest.write(vh_data, 2)  # Write VH to band 2
            dest.write(ratio_data, 3)  # Write ratio to band 3

In [ ]:
# Save as multi-band TIFF
save_as_multiband_tiff(VV_sigma0, VH_sigma0, ratio_VVVH_sigma0, f"{savedir}/rtc", "merged_VV_VH_ratio_sigma0.tif", meta_rtc)

In [ ]:
file_path = f"{savedir}/rtc/merged_VV_VH_ratio_sigma0.tif"  # Path to the multi-band GeoTIFF

# Open the GeoTIFF and get the RGB image
sigma0_rgb = open_geotiff_as_rgb(file_path)

# Plot the RGB image
fig, ax = plt.subplots(1,2, figsize=(12, 12))

cax = ax[0].imshow(gamma0_rgb, cmap='jet', vmin=np.nanpercentile(gamma0_rgb, 2), vmax=np.nanpercentile(gamma0_rgb, 98))
ax[0].set_title("Gamma0 \n (Terrain along LOS + Brightness)", fontsize=12)
ax[0].set_xlabel('Longitude')
ax[0].set_ylabel('Latitude')
ax[0].axis('off')  # Hide axis labels

cax = ax[1].imshow(sigma0_rgb, cmap='jet', vmin=np.nanpercentile(sigma0_rgb, 2), vmax=np.nanpercentile(sigma0_rgb, 98))
ax[1].set_title("Sigma0 \n (Local Terrain + Brightness)", fontsize=12)
ax[1].set_xlabel('Longitude')
ax[1].set_ylabel('Latitude')
ax[1].axis('off')

# fig.suptitle("VV: Red, VH: Green, VV/VH: Blue")

fig.tight_layout(pad=2.0)  # Increase pad value if there is too much space between subplots
# Reduce the top margin to avoid white space above the suptitle
fig.subplots_adjust(top=0.90)

# Convert Gamma0 to Beta0

In [ ]:
## Take the urls of the all the *gamma0_to_sigma0.tif
gamma0_beta0_files = rtc_static_df_filtered['additionalUrls'].apply(lambda x: x[-2] if isinstance(x, (list, str)) else None)

## Download the relevant OPERA RTC Static Layers to a temporary "rtc" folder 
os.makedirs(f'{savedir}/rtc', exist_ok='True')
asf.download_urls(urls=rtc_static_df_filtered.url, path=f'{savedir}/rtc', processes=10)
asf.download_urls(urls=gamma0_beta0_files, path=f'{savedir}/rtc', processes=10)

In [ ]:
# Read, merge, plot the gamma0_to_beta0 static layer.
gamma0_to_beta0, meta_static = read_and_merge_rtc_static(f"{savedir}/rtc/*_rtc_anf_gamma0_to_beta0.tif", f"{savedir}/rtc")

# Plot
fig, ax = plt.subplots(figsize=(12, 12))
cax = ax.imshow(gamma0_to_beta0, cmap='jet', vmin=np.nanpercentile(gamma0_to_beta0, 2), vmax=np.nanpercentile(gamma0_to_beta0, 98))
ax.set_title("Gamm0 to Beta0", fontsize=12)
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
ax.axis('off')  # Hide axis labels

In [ ]:
# Plot each polarization converted from gamma0 to beta0
fig, ax = plt.subplots(1, 3, figsize=(15, 9))

# Gamma0 to beta0 conversion
VV_beta0 = VV[0] * gamma0_to_beta0
VH_beta0 = VH[0] * gamma0_to_beta0
ratio_VVVH_beta0 = VV_beta0 - VH_beta0

# Plot VV
cax1 = ax[0].imshow(VV_beta0, cmap='hot', interpolation=None, origin='upper', extent=bbox, 
                    vmin=np.nanpercentile(VV_beta0, 2), vmax=np.nanpercentile(VV_beta0, 98))
cbar1 = fig.colorbar(cax1, ax=ax[0], orientation='vertical', fraction=0.01, pad=0.02)
cbar1.set_label('VV_beta0', fontsize=12)
ax[0].set_xlabel('Longitude')
ax[0].set_ylabel('Latitude')
ax[0].set_title(f'VV_beta0 - {pd.to_datetime(start_date).date()}', fontsize=12)

# Plot VH
cax2 = ax[1].imshow(VH_beta0, cmap='hot', interpolation=None, origin='upper', extent=bbox, 
                    vmin=np.nanpercentile(VH_beta0, 2), vmax=np.nanpercentile(VH_beta0, 98))
cbar2 = fig.colorbar(cax2, ax=ax[1], orientation='vertical', fraction=0.01, pad=0.02)
cbar2.set_label('VH_beta0', fontsize=12)
ax[1].set_xlabel('Longitude')
ax[1].set_ylabel('Latitude')
ax[1].set_title(f'VH_beta0 - {pd.to_datetime(start_date).date()}', fontsize=12)

# Plot VV/VH Ratio
cax3 = ax[2].imshow(ratio_VVVH_beta0, cmap='hot', interpolation=None, origin='upper', extent=bbox, 
                    vmin=np.nanpercentile(ratio_VVVH_beta0, 2), vmax=np.nanpercentile(ratio_VVVH_beta0, 98))
cbar3 = fig.colorbar(cax3, ax=ax[2], orientation='vertical', fraction=0.01, pad=0.02)
cbar3.set_label('VV/VH Ratio (beta0)', fontsize=12)
ax[2].set_xlabel('Longitude')
ax[2].set_ylabel('Latitude')
ax[2].set_title(f'VV/VH Ratio beta0 - {pd.to_datetime(start_date).date()}', fontsize=12)

plt.tight_layout()  # Adjust subplots to avoid overlap

In [ ]:
# Save as multi-band TIFF
save_as_multiband_tiff(VV_beta0, VH_beta0, ratio_VVVH_beta0, f"{savedir}/rtc", "merged_VV_VH_ratio_beta0.tif", meta_rtc)

In [ ]:
file_path = f"{savedir}/rtc/merged_VV_VH_ratio_beta0.tif"  # Path to the multi-band GeoTIFF

# Open the GeoTIFF and get the RGB image
beta0_rgb = open_geotiff_as_rgb(file_path)

# Plot the RGB image
fig, ax = plt.subplots(1,2, figsize=(12, 12))

cax = ax[0].imshow(gamma0_rgb, cmap='jet', vmin=np.nanpercentile(gamma0_rgb, 2), vmax=np.nanpercentile(gamma0_rgb, 98))
ax[0].set_title("Gamma0 \n (Terrain along LOS + Brightness)", fontsize=12)
ax[0].set_xlabel('Longitude')
ax[0].set_ylabel('Latitude')
ax[0].axis('off')  # Hide axis labels

cax = ax[1].imshow(beta0_rgb, cmap='jet', vmin=np.nanpercentile(beta0_rgb, 2), vmax=np.nanpercentile(beta0_rgb, 98))
ax[1].set_title("Beta0 \n (Brightness Only)", fontsize=12)
ax[1].set_xlabel('Longitude')
ax[1].set_ylabel('Latitude')
ax[1].axis('off')

# fig.suptitle("VV: Red, VH: Green, VV/VH: Blue")

fig.tight_layout(pad=2.0)  # Increase pad value if there is too much space between subplots
# Reduce the top margin to avoid white space above the suptitle
fig.subplots_adjust(top=0.90)